In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import numpy as np
from scipy.ndimage import gaussian_filter1d
from scipy.signal import savgol_filter
import fsspec

# ── Load Data ──────────────────────────────────────────────────────────────────
def load_specific_parquet(url):
    with fsspec.open(url, 'rb') as f:
        return pd.read_parquet(f)

url  = "gs://agntworks-data-dev/wheelsup/processed/df_bookings_metros_final_merge.parquet"
metros_final_merge = load_specific_parquet(url)
Data = metros_final_merge.copy()

#---------------Loading support files------------------------------------------
all_DI_sheets = pd.read_excel('DI_Analysis_LJ_SMID.xlsx', sheet_name=None)
all_LI_sheets = pd.read_excel('LI_Analysis_LJ_SMID.xlsx', sheet_name=None)

#---------------Loading the toggle percent files--------------------------------
all_DI_LI_toggle = pd.read_excel("DI_LI_Analysis_Report with toggle % v2pr.xlsx",sheet_name=None)


In [16]:
#-------------Taking the toggle value for each flight--------------------
DI_Toggle = all_DI_LI_toggle['DI Light SMID']
LI_Toggle = all_DI_LI_toggle['LI Light SMID']

#--------------Adding AirCraft Type column-----------------------------------
all_DI_sheets['DI Light Jet']['AirCraftType'] = 'Light'
all_DI_sheets['DI SMID']['AirCraftType'] = 'Super Midsize'
all_LI_sheets['LI Light Jet']['AirCraftType'] = 'Light'
all_LI_sheets['LI SMID']['AirCraftType'] = 'Super Midsize'

#-----------------Merging both DI and LI data---------------------------------
DI_data = pd.concat([all_DI_sheets['DI Light Jet'], all_DI_sheets['DI SMID']])
LI_data = pd.concat([all_LI_sheets['LI Light Jet'], all_LI_sheets['LI SMID']])

#-------------Renaming the columns Name to match with mapping data------------
DI_data.rename(columns={'Route': 'corridor', 'day_name': 'DOW', 'hour_bucket': 'TOD'}, inplace=True)
LI_data.rename(columns={'Route': 'corridor', 'day_name': 'DOW', 'hour_bucket': 'TOD'}, inplace=True)

#------------Taking only important columns-------------------------------------
# ✅ UPDATED: Using Pct_flight as TI Level, WU_LI_Relative as LI Level (actual values, no ranges)
DI_data = DI_data[['AirCraftType', 'corridor', 'DOW', 'TOD', 'pct_flights']]
LI_data = LI_data[['AirCraftType', 'corridor', 'DOW', 'TOD', 'WU_LI_Relative']]

# ✅ Rename to TI Level and LI Level immediately
DI_data.rename(columns={'pct_flights': 'TI Level'}, inplace=True)
LI_data.rename(columns={'WU_LI_Relative': 'LI Level'}, inplace=True)

# Deduplicate DI and LI lookup tables BEFORE merging
DI_data = DI_data.drop_duplicates(subset=['AirCraftType', 'corridor', 'DOW', 'TOD'], keep='last')
LI_data = LI_data.drop_duplicates(subset=['AirCraftType', 'corridor', 'DOW', 'TOD'], keep='last')

# TOD Mapping
DI_tod_mapping = {
    '07:00-10:00': '07:00-09:59', '10:00-13:00': '10:00-12:59',
    '13:00-16:00': '13:00-15:59', '16:00-19:00': '16:00-18:59',
    '19:00-24:00': '19:00-23:59', '00:00-07:00': '00:00-06:59'
}
LI_tod_mapping = {
    '07:00–10:00': '07:00-09:59', '10:00–13:00': '10:00-12:59',
    '13:00–16:00': '13:00-15:59', '16:00–19:00': '16:00-18:59',
    '19:00–24:00': '19:00-23:59', '00:00–07:00': '00:00-06:59'
}

DI_data['TOD'] = DI_data['TOD'].str.strip().str.replace('–', '-').map(DI_tod_mapping)
LI_data['TOD'] = LI_data['TOD'].map(LI_tod_mapping)

# ✅ UPDATED: Deduplicate Toggle tables on TI Level / LI Level columns
DI_Toggle = DI_Toggle.drop_duplicates(subset=['TI Level', 'AirCraftType'], keep='last')
LI_Toggle = LI_Toggle.drop_duplicates(subset=['LI Level', 'AirCraftType'], keep='last')

#-------------Preparing main flight data---------------
data = Data.copy()
data['corridor'] = data['corridor'].str.replace(' - ', '->')
data.rename(columns={'flightactualAircraftCabinName': 'AirCraftType'}, inplace=True)
data['AirCraftType'] = data['AirCraftType'].map({
    'Super Midsize': 'Super Midsize', 'Premium Super-Mid': 'Super Midsize',
    'Premium Light': 'Light', 'Light': 'Light'
})

map_data = data[['AirCraftType', 'corridor', 'DOW', 'TOD']].copy()


KeyError: Index(['TI Level'], dtype='object')

In [5]:
import pandas as pd
import numpy as np

# ============================================================
# PART 1 — Mapping DI / LI  (Toggle Lookup + Final Toggle)
# ============================================================
print("\n" + "="*60)
print("PART 1 — DI / LI Toggle Mapping")
print("="*60)

# ── Step 1.1: Extract toggle tables ──────────────────────────
DI_Toggle = all_DI_LI_toggle['DI Light SMID'].copy()
LI_Toggle = all_DI_LI_toggle['LI Light SMID'].copy()
print(f"[1.1] DI_Toggle rows : {len(DI_Toggle)} | LI_Toggle rows : {len(LI_Toggle)}")

# ── Step 1.2: Tag AircraftType ────────────────────────────────
all_DI_sheets['DI Light Jet']['AirCraftType'] = 'Light'
all_DI_sheets['DI SMID']['AirCraftType']      = 'Super Midsize'
all_LI_sheets['LI Light Jet']['AirCraftType'] = 'Light'
all_LI_sheets['LI SMID']['AirCraftType']      = 'Super Midsize'
print("[1.2] AirCraftType labels assigned to DI/LI sheets")

# ── Step 1.3: Concat Light + SMID ────────────────────────────
DI_data = pd.concat([all_DI_sheets['DI Light Jet'], all_DI_sheets['DI SMID']], ignore_index=True)
LI_data = pd.concat([all_LI_sheets['LI Light Jet'], all_LI_sheets['LI SMID']], ignore_index=True)
print(f"[1.3] DI_data rows : {len(DI_data)} | LI_data rows : {len(LI_data)}")

# ── Step 1.4: Rename columns ──────────────────────────────────
rename_map = {'Route': 'corridor', 'day_name': 'DOW', 'hour_bucket': 'TOD'}
DI_data.rename(columns=rename_map, inplace=True)
LI_data.rename(columns=rename_map, inplace=True)
print("[1.4] Columns renamed → corridor, DOW, TOD")

# ── Step 1.5: Select & rename value columns ───────────────────
DI_data = (DI_data[['AirCraftType', 'corridor', 'DOW', 'TOD', 'pct_flights']]
           .rename(columns={'pct_flights': 'TI Level'}))
LI_data = (LI_data[['AirCraftType', 'corridor', 'DOW', 'TOD', 'WU_LI_Relative']]
           .rename(columns={'WU_LI_Relative': 'LI Level'}))
print("[1.5] Value columns selected → 'TI Level', 'LI Level'")

# ── Step 1.6: Deduplicate on merge keys ──────────────────────
MERGE_KEYS = ['AirCraftType', 'corridor', 'DOW', 'TOD']
before_di, before_li = len(DI_data), len(LI_data)
DI_data = DI_data.drop_duplicates(subset=MERGE_KEYS, keep='last')
LI_data = LI_data.drop_duplicates(subset=MERGE_KEYS, keep='last')
print(f"[1.6] Dedup  DI_data : {before_di} → {len(DI_data)} rows")
print(f"[1.6] Dedup  LI_data : {before_li} → {len(LI_data)} rows")

# ── Step 1.7: Normalise TOD strings ──────────────────────────
TOD_CANONICAL = {
    '07:00-10:00': '07:00-09:59', '07:00–10:00': '07:00-09:59',
    '10:00-13:00': '10:00-12:59', '10:00–13:00': '10:00-12:59',
    '13:00-16:00': '13:00-15:59', '13:00–16:00': '13:00-15:59',
    '16:00-19:00': '16:00-18:59', '16:00–19:00': '16:00-18:59',
    '19:00-24:00': '19:00-23:59', '19:00–24:00': '19:00-23:59',
    '00:00-07:00': '00:00-06:59', '00:00–07:00': '00:00-06:59',
}
DI_data['TOD'] = DI_data['TOD'].str.strip().str.replace('–', '-', regex=False).map(TOD_CANONICAL)
LI_data['TOD'] = LI_data['TOD'].str.strip().str.replace('–', '-', regex=False).map(TOD_CANONICAL)
di_tod_nulls = DI_data['TOD'].isna().sum()
li_tod_nulls = LI_data['TOD'].isna().sum()
print(f"[1.7] TOD normalised  | DI unmapped : {di_tod_nulls} | LI unmapped : {li_tod_nulls}")

# ── Step 1.8: Build range-bucket helper ──────────────────────
def map_to_range_bucket(value):
    """Float → 'floor - (floor+1)' string, e.g. 2.3 → '2 - 3'"""
    if pd.isna(value):
        return None
    floor = int(value)
    return f"{floor} - {floor + 1}"

DI_data['_TI_bucket'] = DI_data['TI Level'].apply(map_to_range_bucket)
LI_data['_LI_bucket'] = LI_data['LI Level'].apply(map_to_range_bucket)
print("[1.8] Range buckets created (_TI_bucket, _LI_bucket)")

# ── Step 1.9: Dedup toggle tables ────────────────────────────
before_dt, before_lt = len(DI_Toggle), len(LI_Toggle)
DI_Toggle = DI_Toggle.drop_duplicates(subset=['Tier_gap',  'AirCraftType'], keep='last')
LI_Toggle = LI_Toggle.drop_duplicates(subset=['WU_LI_gap', 'AirCraftType'], keep='last')
print(f"[1.9] Toggle dedup  DI_Toggle : {before_dt} → {len(DI_Toggle)} rows")
print(f"[1.9] Toggle dedup  LI_Toggle : {before_lt} → {len(LI_Toggle)} rows")

# ── Step 1.10: Join toggles via bucket key ───────────────────
DI_lookup = (
    DI_data
    .merge(
        DI_Toggle[['Tier_gap', 'AirCraftType', 'DI Toggle']],
        left_on=['_TI_bucket', 'AirCraftType'],
        right_on=['Tier_gap',  'AirCraftType'],
        how='left'
    )
    .drop(columns=['_TI_bucket', 'Tier_gap'])
    .rename(columns={'DI Toggle': 'TI Toggle'})
)

LI_lookup = (
    LI_data
    .merge(
        LI_Toggle[['WU_LI_gap', 'AirCraftType', 'LI Toggle']],
        left_on=['_LI_bucket', 'AirCraftType'],
        right_on=['WU_LI_gap',  'AirCraftType'],
        how='left'
    )
    .drop(columns=['_LI_bucket', 'WU_LI_gap'])
    .rename(columns={'LI Toggle': 'LI Toggle'})
)

di_unmatched = DI_lookup['TI Toggle'].isna().sum()
li_unmatched = LI_lookup['LI Toggle'].isna().sum()
print(f"[1.10] DI_lookup rows : {len(DI_lookup)} | unmatched TI Toggle : {di_unmatched}")
print(f"[1.10] LI_lookup rows : {len(LI_lookup)} | unmatched LI Toggle : {li_unmatched}")

# ── Step 1.11: Combine DI + LI lookups ──────────────────────
combined_lookup = pd.merge(
    DI_lookup,
    LI_lookup,
    on=MERGE_KEYS,
    how='outer'
)
print(f"[1.11] combined_lookup rows : {len(combined_lookup)}")

# ── Step 1.12: Prepare main flight data ─────────────────────
data = Data.copy()
data['corridor'] = data['corridor'].str.replace(' - ', '->', regex=False)
data.rename(columns={'flightactualAircraftCabinName': 'AirCraftType'}, inplace=True)
data['AirCraftType'] = data['AirCraftType'].map({
    'Super Midsize':     'Super Midsize',
    'Premium Super-Mid': 'Super Midsize',
    'Premium Light':     'Light',
    'Light':             'Light',
})
unmapped_act = data['AirCraftType'].isna().sum()
print(f"[1.12] Flight data rows : {len(data)} | unmapped AirCraftType : {unmapped_act}")

# ── Step 1.13: Index-join (prevents row explosion) ───────────
combined_lookup_idx = combined_lookup.set_index(MERGE_KEYS)
joined = (
    data[MERGE_KEYS]
    .set_index(MERGE_KEYS)
    .join(combined_lookup_idx, how='left')
)
print(f"[1.13] Joined rows : {len(joined)} (expected {len(data)})")
assert len(joined) == len(data), "⚠️  Row count mismatch after index-join!"

# ── Step 1.14: Weights + Final Toggle (Part-1 version) ───────
joined['WT_TI'] = 0.6
joined['WT_LI'] = 0.1
joined['Final Toggle'] = (
    joined['WT_TI'] * joined['TI Toggle'] +
    joined['WT_LI'] * joined['LI Toggle']
)
print(f"[1.14] Final Toggle  | NaN : {joined['Final Toggle'].isna().sum()}")

# ── Step 1.15: Attach back + Toggle Delta ───────────────────
final_data = pd.concat(
    [data.reset_index(drop=True),
     joined[['TI Level', 'LI Level', 'TI Toggle', 'LI Toggle',
             'WT_TI', 'WT_LI', 'Final Toggle']].reset_index(drop=True)],
    axis=1
)
final_data['Toggle Delta'] = (
    final_data['flightcost'] / (1 + final_data['final_adjustment_pct'] / 100)
) * (final_data['Final Toggle'] - final_data['final_adjustment_pct'] / 100)

assert len(final_data) == len(data), "⚠️  Row count mismatch in final_data!"
print(f"[1.15] ✅ final_data : {len(final_data)} rows × {len(final_data.columns)} columns")
print(f"       Toggle Delta NaN : {final_data['Toggle Delta'].isna().sum()}")


# ============================================================
# PART 2 — Seasonality Index (SI) Integration
# ============================================================
print("\n" + "="*60)
print("PART 2 — Seasonality Index (SI) Integration")
print("="*60)

# ── Step 2.1: Select required columns ───────────────────────
REQUIRED_COLS = [
    'corridor', 'AirCraftType', 'DOW', 'TOD',
    'TI Level', 'LI Level',
    'flightEstimatedDepartureTime_ET',
    'TI Toggle', 'LI Toggle', 'WT_TI', 'WT_LI',
    'Final Toggle', 'Toggle Delta',
    'flightcost', 'final_adjustment_pct'
]
req_data = final_data[REQUIRED_COLS].copy()
req_data['month'] = req_data['flightEstimatedDepartureTime_ET'].dt.month
req_data.drop(columns=['flightEstimatedDepartureTime_ET'], inplace=True)
print(f"[2.1] req_data : {len(req_data)} rows | month range : {req_data['month'].min()}–{req_data['month'].max()}")

# ── Step 2.2: Load & clean seasonality CSV ──────────────────
si_data = pd.read_csv("lj_smid_kmean_intensity_slab_va.csv")
print(f"[2.2] SI raw rows : {len(si_data)}")

MONTH_MAP = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}
si_data['month'] = si_data['month'].map(MONTH_MAP)
si_data['corridor'] = si_data['corridor'].str.replace(' -> ', '->', regex=False)
si_data.rename(columns={'AircraftType': 'AirCraftType'}, inplace=True)
si_data['AirCraftType'] = si_data['AirCraftType'].map({
    'Light Jet':         'Light',
    'Super Midsize Jet': 'Super Midsize',
    'Super Midsize':     'Super Midsize',
    'Light':             'Light',
})
print(f"[2.2] SI after normalisation : {len(si_data)} rows | AirCraftType nulls : {si_data['AirCraftType'].isna().sum()}")

# ── Step 2.3: Select SI columns + strict dedup ──────────────
SI_KEYS = ['AirCraftType', 'corridor', 'month']
si_data_to_join = (
    si_data[SI_KEYS + ['density_pct', 'cluster_label', 'slab', 'intensity']]
    .copy()
    .sort_values(SI_KEYS)
    .drop_duplicates(subset=SI_KEYS, keep='last')
    .reset_index(drop=True)
)
print(f"[2.3] SI after dedup : {len(si_data_to_join)} rows (1 per AirCraftType × corridor × month)")

# ── Step 2.4: Merge SI into req_data ────────────────────────
final_result = pd.merge(
    req_data,
    si_data_to_join,
    on=SI_KEYS,
    how='left',
    validate='m:1'   # raises MergeError if SI side has duplicates
)
assert len(final_result) == len(req_data), (
    f"⚠️  Row explosion! req_data={len(req_data)}, final_result={len(final_result)}"
)
si_present_mask = final_result['intensity'].notna()
print(f"[2.4] SI merge complete : {len(final_result)} rows")
print(f"      SI present : {si_present_mask.sum():,} | SI missing : {(~si_present_mask).sum():,}")
final_result.rename(columns={'intensity': 'SI_Toggle'}, inplace=True)

# ── Step 2.5: Dynamic weights ────────────────────────────────
final_result['WT_TI'] = np.where(si_present_mask, 0.6, 0.8)
final_result['WT_LI'] = np.where(si_present_mask, 0.1, 0.2)
final_result['WT_SI'] = np.where(si_present_mask, 0.3, 0.0)
print("[2.5] Weights assigned  SI present → (0.6, 0.1, 0.3) | SI missing → (0.8, 0.2, 0.0)")

# ── Step 2.6: Weighted intensity score ──────────────────────
final_result['final_intensity'] = np.where(
    si_present_mask,
    final_result['WT_TI'] * final_result['TI Level'] +
    final_result['WT_LI'] * final_result['LI Level'] +
    final_result['WT_SI'] * final_result['SI_Toggle'],
    final_result['WT_TI'] * final_result['TI Level'] +
    final_result['WT_LI'] * final_result['LI Level']
)
print(f"[2.6] final_intensity  | NaN : {final_result['final_intensity'].isna().sum()} "
      f"| min : {final_result['final_intensity'].min():.4f} "
      f"| max : {final_result['final_intensity'].max():.4f}")

# ── Step 2.7: Intensity → Final Toggle mapping ───────────────
def apply_intensity_to_toggle(series: pd.Series) -> np.ndarray:
    """Bin final_intensity into toggle values."""
    conditions = [
        series < 2,
        (series >= 2) & (series < 3),
        (series >= 3) & (series < 4),
        (series >= 4) & (series < 5),
        (series >= 5) & (series < 6),
        (series >= 6) & (series < 7),
        (series >= 7) & (series < 8),
        series >= 8,
    ]
    choices = [0.00, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
    return np.select(conditions, choices, default=np.nan)

final_result['Final Toggle'] = apply_intensity_to_toggle(final_result['final_intensity'])
nan_ft = np.isnan(final_result['Final Toggle']).sum()
print(f"[2.7] Final Toggle mapped | NaN : {nan_ft} | distribution :")
print(final_result['Final Toggle'].value_counts(dropna=False).sort_index().to_string())

# ── Step 2.8: Toggle Delta ───────────────────────────────────
final_result['Toggle Delta'] = (
    final_result['flightcost'] / (1 + final_result['final_adjustment_pct'] / 100)
) * (final_result['Final Toggle'] - final_result['final_adjustment_pct'] / 100)
print(f"[2.8] Toggle Delta  | NaN : {final_result['Toggle Delta'].isna().sum()}")

# ── Step 2.9: Merge new columns back into final_data ─────────
OVERWRITE_COLS = [
    'WT_TI', 'WT_LI', 'WT_SI',
    'Final Toggle', 'Toggle Delta',
    'SI_Toggle', 'final_intensity',
    'density_pct', 'cluster_label', 'slab', 'month'
]
# Drop stale columns from final_data then concat fresh ones
final_merged = final_data.drop(
    columns=[c for c in OVERWRITE_COLS if c in final_data.columns],
    errors='ignore'
).reset_index(drop=True)

new_cols_only = [c for c in final_result.columns if c not in final_merged.columns]
final_merged = pd.concat(
    [final_merged,
     final_result[new_cols_only + OVERWRITE_COLS].reset_index(drop=True)],
    axis=1
)
assert len(final_merged) == len(final_data), "⚠️  Row count mismatch after Part-2 merge!"
print(f"\n[2.9] ✅ final_merged : {len(final_merged)} rows × {len(final_merged.columns)} columns")
print(f"      SI present  : {si_present_mask.sum():,}  (WT_TI=0.6 | WT_LI=0.1 | WT_SI=0.3)")
print(f"      SI missing  : {(~si_present_mask).sum():,}  (WT_TI=0.8 | WT_LI=0.2 | WT_SI=0.0)")
print(f"      NaN Final Toggle : {final_merged['Final Toggle'].isna().sum()}")
print(f"      NaN Toggle Delta : {final_merged['Toggle Delta'].isna().sum()}")


# ============================================================
# PART 3 — TI / LI / SI Corridor Grids
# ============================================================
print("\n" + "="*60)
print("PART 3 — Corridor Grids (TI × DOW/TOD | LI × DOW/TOD | SI scalar)")
print("="*60)

TOD_ORDER = ['00:00-06:59', '07:00-09:59', '10:00-12:59',
             '13:00-15:59', '16:00-18:59', '19:00-23:59']
DOW_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday',
             'Friday', 'Saturday', 'Sunday']

def make_grid(df, index_col, col_col, value_col, label, index_order, col_order):
    """Pivot: rows=index_col, cols=col_col, values=mean(value_col)."""
    return (
        df.groupby([index_col, col_col])[value_col]
          .mean()
          .reset_index()
          .pivot(index=index_col, columns=col_col, values=value_col)
          .reindex(index=index_order, columns=col_order)
          .round(4)
          .rename_axis(index=label, columns=None)
    )

# ── Step 3.1: Build deduplicated source tables ───────────────
ti_source = (
    DI_lookup[['AirCraftType', 'corridor', 'DOW', 'TOD', 'TI Level']]
    .drop_duplicates(subset=MERGE_KEYS, keep='last')
    .copy()
)
li_source = (
    LI_lookup[['AirCraftType', 'corridor', 'DOW', 'TOD', 'LI Level']]
    .drop_duplicates(subset=MERGE_KEYS, keep='last')
    .copy()
)
target_month = final_result['month'].mode()[0]
si_source = (
    si_data_to_join[si_data_to_join['month'] == target_month]
    [['AirCraftType', 'corridor', 'intensity']]
    .drop_duplicates(subset=['AirCraftType', 'corridor'], keep='last')
    .copy()
)
print(f"[3.1] ti_source : {len(ti_source)} rows | li_source : {len(li_source)} rows "
      f"| si_source (month={target_month}) : {len(si_source)} rows")

# ── Step 3.2: Define corridors ───────────────────────────────
CORRIDORS = [
    'Denver->LA Basin | Light',
    'Charlotte->South Florida | Light',
    'Chicago->South Florida | Light',
    'LA Basin->Denver | Light',
    'South Florida->Charlotte | Light',
    'South Florida->Chicago | Light',
    'New York City->South Florida | Super Midsize',
    'South Florida->New York City | Super Midsize',
    'Denver->South Florida | Super Midsize',
    'South Florida->Denver | Super Midsize',
    'Chicago->South Florida | Super Midsize',
    'South Florida->Chicago | Super Midsize',
]
print(f"[3.2] Processing {len(CORRIDORS)} corridors …")

# ── Step 3.3: Build grids ────────────────────────────────────
corridor_grids = {}
empty_ti = empty_li = si_missing = 0

for corridor_key in CORRIDORS:
    corridor, actype = corridor_key.rsplit(' | ', 1)
    mask_base = {'corridor': corridor, 'AirCraftType': actype}

    # TI grid
    ti_grp = ti_source.query("corridor == @corridor and AirCraftType == @actype")
    if not ti_grp.empty:
        ti_grid = make_grid(ti_grp, 'TOD', 'DOW', 'TI Level', 'TI', TOD_ORDER, DOW_ORDER)
    else:
        ti_grid = pd.DataFrame(index=pd.Index(TOD_ORDER, name='TI'), columns=DOW_ORDER)
        empty_ti += 1

    # LI grid
    li_grp = li_source.query("corridor == @corridor and AirCraftType == @actype")
    if not li_grp.empty:
        li_grid = make_grid(li_grp, 'TOD', 'DOW', 'LI Level', 'LI', TOD_ORDER, DOW_ORDER)
    else:
        li_grid = pd.DataFrame(index=pd.Index(TOD_ORDER, name='LI'), columns=DOW_ORDER)
        empty_li += 1

    # SI grid (scalar broadcast)
    si_row = si_source.query("corridor == @corridor and AirCraftType == @actype")
    si_val = si_row['intensity'].values[0] if not si_row.empty else np.nan
    if si_row.empty:
        si_missing += 1
    si_grid = pd.DataFrame(
        si_val,
        index=pd.Index(TOD_ORDER, name=f'SI (Month {target_month})'),
        columns=DOW_ORDER
    ).round(4)

    corridor_grids[corridor_key] = {'TI': ti_grid, 'LI': li_grid, 'SI': si_grid}

print(f"[3.3] Grids built : {len(corridor_grids)} total | "
      f"empty TI : {empty_ti} | empty LI : {empty_li} | missing SI : {si_missing}")

# ── Step 3.4: Filter to target corridors & sanity check ──────
filtered_corridor_grids = {k: corridor_grids[k] for k in CORRIDORS if k in corridor_grids}
print(f"[3.4] Filtered grids : {len(filtered_corridor_grids)} / {len(CORRIDORS)}")

# ── Step 3.5: Sample output ──────────────────────────────────
SAMPLE = 'New York City->South Florida | Super Midsize'
if SAMPLE in filtered_corridor_grids:
    print(f"\n[3.5] Sample — {SAMPLE}")
    print("\n  TI Grid:\n", filtered_corridor_grids[SAMPLE]['TI'].to_string())
    print("\n  LI Grid:\n", filtered_corridor_grids[SAMPLE]['LI'].to_string())
    print("\n  SI Grid:\n", filtered_corridor_grids[SAMPLE]['SI'].to_string())
else:
    print(f"[3.5] ⚠️  Sample corridor not found : {SAMPLE}")

print("\n" + "="*60)
print("✅ ALL PARTS COMPLETE")
print("="*60)


PART 1 — DI / LI Toggle Mapping
[1.1] DI_Toggle rows : 22 | LI_Toggle rows : 22
[1.2] AirCraftType labels assigned to DI/LI sheets
[1.3] DI_data rows : 38052 | LI_data rows : 38682
[1.4] Columns renamed → corridor, DOW, TOD
[1.5] Value columns selected → 'TI Level', 'LI Level'
[1.6] Dedup  DI_data : 38052 → 38052 rows
[1.6] Dedup  LI_data : 38682 → 38682 rows
[1.7] TOD normalised  | DI unmapped : 0 | LI unmapped : 0
[1.8] Range buckets created (_TI_bucket, _LI_bucket)
[1.9] Toggle dedup  DI_Toggle : 22 → 22 rows
[1.9] Toggle dedup  LI_Toggle : 22 → 22 rows
[1.10] DI_lookup rows : 38052 | unmatched TI Toggle : 168
[1.10] LI_lookup rows : 38682 | unmatched LI Toggle : 295
[1.11] combined_lookup rows : 38682
[1.12] Flight data rows : 115071 | unmapped AirCraftType : 0
[1.13] Joined rows : 115071 (expected 115071)
[1.14] Final Toggle  | NaN : 6873
[1.15] ✅ final_data : 115071 rows × 60 columns
       Toggle Delta NaN : 7070

PART 2 — Seasonality Index (SI) Integration
[2.1] req_data : 115

In [6]:
final_result[(final_result['corridor']=='Charlotte->South Florida')&(final_result['AirCraftType']=='Light')]

,corridor,AirCraftType,DOW,TOD,TI Level,LI Level,TI Toggle,LI Toggle,WT_TI,WT_LI,...,Toggle Delta,flightcost,final_adjustment_pct,month,density_pct,cluster_label,slab,SI_Toggle,WT_SI,final_intensity
40,Charlotte->South Florida,Light,Monday,10:00-12:59,4.620732,3.281960,0.25,0.15,0.6,0.1,...,-957.600000,11970.00,25.0,6,7.12,winter-spring-peak,6%-9%,2.0,0.3,3.700635
62,Charlotte->South Florida,Light,Monday,10:00-12:59,4.620732,3.281960,0.25,0.15,0.6,0.1,...,-787.200000,9840.00,25.0,6,7.12,winter-spring-peak,6%-9%,2.0,0.3,3.700635
154,Charlotte->South Florida,Light,Tuesday,10:00-12:59,3.312206,2.287891,0.15,0.10,0.6,0.1,...,-575.200000,13229.60,15.0,5,8.43,winter-spring-peak,6%-9%,2.0,0.3,2.816113
187,Charlotte->South Florida,Light,Saturday,07:00-09:59,2.657943,3.092424,0.00,0.15,0.6,0.1,...,-509.630435,11721.50,15.0,6,7.12,winter-spring-peak,6%-9%,2.0,0.3,2.504008
226,Charlotte->South Florida,Light,Sunday,13:00-15:59,4.988755,2.705871,0.25,0.10,0.6,0.1,...,133.043478,3060.00,15.0,4,9.75,winter-spring-peak,9%-12%,3.0,0.3,4.163840
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114374,Charlotte->South Florida,Light,Saturday,07:00-09:59,2.657943,3.092424,0.00,0.15,0.6,0.1,...,-652.702174,15012.15,15.0,11,9.00,winter-spring-peak,9%-12%,3.0,0.3,2.804008
114542,Charlotte->South Florida,Light,Sunday,19:00-23:59,0.613371,2.042167,0.00,0.10,0.6,0.1,...,0.000000,15998.80,0.0,4,9.75,winter-spring-peak,9%-12%,3.0,0.3,1.472240
114698,Charlotte->South Florida,Light,Sunday,19:00-23:59,0.613371,2.042167,0.00,0.10,0.6,0.1,...,0.000000,15998.80,0.0,4,9.75,winter-spring-peak,9%-12%,3.0,0.3,1.472240
115006,Charlotte->South Florida,Light,Wednesday,13:00-15:59,3.700675,1.738713,0.15,0.05,0.6,0.1,...,1765.575000,11770.50,0.0,4,9.75,winter-spring-peak,9%-12%,3.0,0.3,3.294276


In [ ]:
final_merged.to_excel("Final_Data_new_with_filter.xlsx",index=False)

In [66]:
filtered_corridor_grids['Charlotte->South Florida | Light']

{'TI':              Monday  Tuesday  Wednesday  Thursday  Friday  Saturday  Sunday
 TI                                                                         
 00:00-06:59  0.4296   0.3556     0.3259    0.3852  0.2370    0.2074  0.2370
 07:00-09:59  3.2000   2.8296     3.0222    3.1704  2.9333    2.6963  2.8296
 10:00-12:59  4.5926   3.6444     3.4667    3.7630  4.2963    4.2963  4.9926
 13:00-15:59  3.5704   3.3778     3.6889    4.0296  4.2667    2.8000  4.9778
 16:00-18:59  2.1481   2.1333     2.4296    2.5481  2.4741    1.7926  2.9481
 19:00-23:59  0.5333   0.5630     0.7259    0.8296  0.7407    0.7259  0.7852,
 'LI':              Monday  Tuesday  Wednesday  Thursday  Friday  Saturday  Sunday
 LI                                                                         
 00:00-06:59  1.4929   2.7059     1.9679    1.6652  1.3529    3.0924  4.0588
 07:00-09:59  1.9041   1.7000     1.8039    2.3265  2.0772    3.0924  2.9467
 10:00-12:59  3.2820   2.2879     2.7753    2.5567  2.9858    3

In [8]:
# ── Write to Excel — one sheet per corridor ──────────────────────────────────
output_path = 'corridor_TI_LI_SI_grids.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for key, grids in filtered_corridor_grids.items():
        sheet_name = key[:31]   # Excel 31-char sheet limit
        startrow   = 0

        for label, grid in grids.items():
            grid.to_excel(writer, sheet_name=sheet_name, startrow=startrow)
            startrow += len(grid) + 3   # 3-row gap between TI / LI / SI blocks

print(f"✅ Written: {output_path}")
print(f"   Corridors: {len(corridor_grids)}")

✅ Written: corridor_TI_LI_SI_grids.xlsx
   Corridors: 12
